# LSTM - GA Optimization

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
import numpy as np
import json
import time
from collections import Counter
from data_loader import load_clinc
from ga_optimizer import GeneticOptimizer

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

USE_SAMPLE = True
data = load_clinc(use_sample=USE_SAMPLE)
train_texts, train_labels = data['train_texts'], data['train_labels']
val_texts, val_labels = data['val_texts'], data['val_labels']
test_texts, test_labels = data['test_texts'], data['test_labels']
num_classes = data['num_classes']

cuda
1525 / 620 / 1100, 151 classes (sample)


In [3]:
class Vocabulary:
    def __init__(self, min_freq=2):
        self.word2idx = {'<PAD>': 0, '<UNK>': 1}
        self.idx2word = {0: '<PAD>', 1: '<UNK>'}
        self.min_freq = min_freq

    def build_vocab(self, texts):
        counts = Counter()
        for t in texts:
            counts.update(t.lower().split())
        idx = 2
        for word, cnt in counts.items():
            if cnt >= self.min_freq:
                self.word2idx[word] = idx
                self.idx2word[idx] = word
                idx += 1

    def encode(self, text):
        return [self.word2idx.get(w, 1) for w in text.lower().split()]

    def __len__(self):
        return len(self.word2idx)


vocab = Vocabulary(min_freq=2)
vocab.build_vocab(train_texts)
vocab_size = len(vocab)
print(f"Vocab: {vocab_size}")

Vocab: 876


In [4]:
class TextSequenceDataset(Dataset):
    def __init__(self, texts, labels, vocab):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return torch.LongTensor(self.vocab.encode(self.texts[idx])), torch.LongTensor([self.labels[idx]])


def collate_batch(batch):
    seqs, labels = zip(*batch)
    return pad_sequence(seqs, batch_first=True, padding_value=0), torch.LongTensor([len(s) for s in seqs]), torch.cat(labels)


class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes,
                 num_layers=2, dropout=0.3, bidirectional=True):
        super().__init__()
        self.bidirectional = bidirectional
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers,
                           batch_first=True, dropout=dropout if num_layers > 1 else 0,
                           bidirectional=bidirectional)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * (2 if bidirectional else 1), num_classes)

    def forward(self, sequences, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(
            self.embedding(sequences), lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (hidden, _) = self.lstm(packed)
        if self.bidirectional:
            hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        else:
            hidden = hidden[-1]
        return self.fc(self.dropout(hidden))

In [5]:
PATIENCE = 5
MAX_EPOCHS = 30

train_ds = TextSequenceDataset(train_texts, train_labels, vocab)
val_ds = TextSequenceDataset(val_texts, val_labels, vocab)


def evaluate_hyperparams(genes):
    torch.manual_seed(42)
    np.random.seed(42)

    tr_loader = DataLoader(train_ds, batch_size=genes['batch_size'], shuffle=True, collate_fn=collate_batch)
    vl_loader = DataLoader(val_ds, batch_size=genes['batch_size'], shuffle=False, collate_fn=collate_batch)

    model = LSTMClassifier(
        vocab_size, genes['embedding_dim'], genes['hidden_dim'], num_classes,
        num_layers=genes['num_layers'], dropout=genes['dropout'],
        bidirectional=True
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=genes['learning_rate'])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

    best_f1, patience_counter = 0, 0

    for epoch in range(MAX_EPOCHS):
        model.train()
        for seqs, lengths, labels in tr_loader:
            seqs, lengths, labels = seqs.to(device), lengths.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(seqs, lengths), labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

        model.eval()
        all_preds = []
        with torch.no_grad():
            for seqs, lengths, labels in vl_loader:
                seqs, lengths = seqs.to(device), lengths.to(device)
                all_preds.extend(model(seqs, lengths).argmax(1).cpu().numpy())

        vl_f1 = f1_score(val_labels, all_preds, average='macro', zero_division=0)
        scheduler.step(vl_f1)

        if vl_f1 > best_f1:
            best_f1, patience_counter = vl_f1, 0
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            break

    return best_f1, {"val_f1": round(best_f1, 4), "epochs": epoch + 1}

In [6]:
search_space = {
    'embedding_dim': [64, 128, 256],
    'hidden_dim': [128, 256, 512],
    'num_layers': [1, 2, 3],
    'dropout': [0.1, 0.2, 0.3, 0.4, 0.5],
    'learning_rate': [0.0001, 0.0005, 0.001, 0.005],
    'batch_size': [32, 64, 128]
}

ga = GeneticOptimizer(
    search_space=search_space,
    fitness_fn=evaluate_hyperparams,
    population_size=15,
    n_generations=10,
    crossover_rate=0.8,
    mutation_rate=0.2,
    tournament_size=3,
    elitism_count=2,
    seed=42
)

ga_results = ga.run()
print(f"\nBest F1: {ga_results['best_fitness']:.4f}")
print(f"Best params: {ga_results['best_genes']}")
print(f"Total evaluations: {ga_results['total_evaluations']}")
print(f"Total time: {ga_results['total_time_seconds']:.1f}s")

Gen  1/10  best=0.5550  avg=0.4534  global_best=0.5550
Gen  2/10  best=0.5550  avg=0.5101  global_best=0.5550
Gen  3/10  best=0.5552  avg=0.5288  global_best=0.5552
Gen  4/10  best=0.5552  avg=0.5174  global_best=0.5552
Gen  5/10  best=0.5605  avg=0.5070  global_best=0.5605
Gen  6/10  best=0.5605  avg=0.5155  global_best=0.5605
Gen  7/10  best=0.5605  avg=0.5505  global_best=0.5605
Gen  8/10  best=0.5605  avg=0.5171  global_best=0.5605
Gen  9/10  best=0.5806  avg=0.5421  global_best=0.5806
Gen 10/10  best=0.5806  avg=0.5297  global_best=0.5806

Best F1: 0.5806
Best params: {'embedding_dim': 256, 'hidden_dim': 128, 'num_layers': 1, 'dropout': 0.5, 'learning_rate': 0.001, 'batch_size': 32}
Total evaluations: 145
Total time: 3155.1s


In [7]:
best = ga_results['best_genes']
torch.manual_seed(42)
np.random.seed(42)

tr_loader = DataLoader(train_ds, batch_size=best['batch_size'], shuffle=True, collate_fn=collate_batch)
vl_loader = DataLoader(val_ds, batch_size=best['batch_size'], shuffle=False, collate_fn=collate_batch)
te_loader = DataLoader(TextSequenceDataset(test_texts, test_labels, vocab),
                       batch_size=best['batch_size'], shuffle=False, collate_fn=collate_batch)

model = LSTMClassifier(
    vocab_size, best['embedding_dim'], best['hidden_dim'], num_classes,
    num_layers=best['num_layers'], dropout=best['dropout'], bidirectional=True
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=best['learning_rate'])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

best_f1, best_state, patience_counter = 0, None, 0

for epoch in range(MAX_EPOCHS):
    model.train()
    for seqs, lengths, labels in tr_loader:
        seqs, lengths, labels = seqs.to(device), lengths.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(seqs, lengths), labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

    model.eval()
    vl_preds = []
    with torch.no_grad():
        for seqs, lengths, labels in vl_loader:
            vl_preds.extend(model(seqs.to(device), lengths.to(device)).argmax(1).cpu().numpy())

    vl_f1 = f1_score(val_labels, vl_preds, average='macro', zero_division=0)
    scheduler.step(vl_f1)

    if vl_f1 > best_f1:
        best_f1, best_state, patience_counter = vl_f1, model.state_dict().copy(), 0
    else:
        patience_counter += 1
    if patience_counter >= PATIENCE:
        break

model.load_state_dict(best_state)
model.eval()
te_preds = []
with torch.no_grad():
    for seqs, lengths, labels in te_loader:
        te_preds.extend(model(seqs.to(device), lengths.to(device)).argmax(1).cpu().numpy())

acc = accuracy_score(test_labels, te_preds)
p = precision_score(test_labels, te_preds, average='macro', zero_division=0)
r = recall_score(test_labels, te_preds, average='macro', zero_division=0)
f1_m = f1_score(test_labels, te_preds, average='macro', zero_division=0)
f1_w = f1_score(test_labels, te_preds, average='weighted', zero_division=0)

print(f"Accuracy:   {acc:.4f}")
print(f"Precision:  {p:.4f}")
print(f"Recall:     {r:.4f}")
print(f"F1 macro:   {f1_m:.4f}")
print(f"F1 weighted:{f1_w:.4f}")

Accuracy:   0.5564
Precision:  0.5973
Recall:     0.5856
F1 macro:   0.5599
F1 weighted:0.5468


In [8]:
results = {
    "model_type": "LSTM",
    "optimization": "genetic_algorithm",
    "best_hyperparameters": ga_results['best_genes'],
    "total_evaluations": ga_results['total_evaluations'],
    "search_time_seconds": ga_results['total_time_seconds'],
    "test_metrics": {
        "accuracy": round(acc, 4),
        "macro_precision": round(p, 4),
        "macro_recall": round(r, 4),
        "macro_f1": round(f1_m, 4),
        "weighted_f1": round(f1_w, 4)
    },
    "ga_params": {
        "population_size": 15,
        "n_generations": 10,
        "crossover_rate": 0.8,
        "mutation_rate": 0.2,
        "tournament_size": 3,
        "elitism_count": 2
    }
}

with open('results/results_lstm_ga.json', 'w') as f:
    json.dump(results, f, indent=4, default=str)

ga.save_results('results/results_lstm_ga_full.json')

print("Saved: results/results_lstm_ga.json, results/results_lstm_ga_full.json")

Saved: results/results_lstm_ga.json, results/results_lstm_ga_full.json
